In [2]:
import numpy as np
from scipy.linalg import expm
from scipy.optimize import minimize

# ---------------------------------------------------------------
# 1. Geometry and potential shapes (first-order Born matrix elements)
# ---------------------------------------------------------------
k = 1.0
n1 = np.array([0.0, 0.0, 1.0])
n2 = np.array([1.0, 0.0, 0.0])
q = k * (n2 - n1)
qz = q[2]
q2 = q @ q

def iso_gaussian_hat(a):
    """FT of exp(-r^2/2a^2): d = value at q=0, g = value at q."""
    pref = (2 * np.pi * a**2) ** 1.5
    d = pref
    g = pref * np.exp(-q2 * a**2 / 2.0)
    return d, g

def dipole_gaussian_hat(a):
    """FT of z*exp(-r^2/2a^2) = i d/dqz [iso FT]; d(0)=0 by oddness."""
    d0, g0 = iso_gaussian_hat(a)
    g = 1j * (-qz * a**2) * g0
    return 0.0, g

a1, a2v = 1.0, 0.5
d1a, g1a = iso_gaussian_hat(a1)
d1b, g1b = iso_gaussian_hat(a2v)
d2, g2 = dipole_gaussian_hat(a1)

print("Channel v1a (isotropic, a=1.0):   d=%.4f  g=%.4f" % (d1a, g1a.real if np.isreal(g1a) else g1a))
print("Channel v1b (isotropic, a=0.5):   d=%.4f  g=%.4f" % (d1b, g1b.real if np.isreal(g1b) else g1b))
print("Channel v2  (dipole,   a=1.0):    d=%.4f  g=%s" % (d2, g2))

I2 = np.eye(2, dtype=complex)
sx = np.array([[0, 1], [1, 0]], dtype=complex)
sy = np.array([[0, -1j], [1j, 0]], dtype=complex)
sz = np.array([[1, 0], [0, -1]], dtype=complex)

V1a = d1a * I2 + g1a.real * sx
V1b = d1b * I2 + g1b.real * sx
# g2 is purely imaginary = i*b -> matrix [[0,i b],[-i b,0]] = -b*sy
b = g2.imag
V2 = -b * sy

print("\nV1a =\n", V1a)
print("V1b =\n", V1b)
print("V2  =\n", V2)

# Linear map from physical channel strengths (eps_1a, eps_1b, eps_2)
# to control triad (c0, cx, cy):
#   c0 = d1a*eps_1a + d1b*eps_1b
#   cx = g1a*eps_1a + g1b*eps_1b
#   cy = -b*eps_2
M = np.array([[d1a, d1b], [g1a.real, g1b.real]])
print("\nCalibration matrix M (c0,cx) = M (eps_1a,eps_1b):\n", M, "\ndet(M)=", np.linalg.det(M))

# ---------------------------------------------------------------
# 2. Target gate: single-qubit Hadamard
# ---------------------------------------------------------------
Ud = (1 / np.sqrt(2)) * np.array([[1, 1], [1, -1]], dtype=complex)
Ud_norm2 = np.trace(Ud.conj().T @ Ud).real  # = 2

# ---------------------------------------------------------------
# 3. Piecewise-constant control optimization
# ---------------------------------------------------------------
K = 6  # number of time bins (fixed)

def build_U(params, T):
    """params: length 3K array of (c0_k, cx_k, cy_k) per bin, time-ordered k=0..K-1 applied in sequence."""
    dt = T / K
    U = np.eye(2, dtype=complex)
    for kk in range(K):
        c0, cx, cy = params[3*kk:3*kk+3]
        H = c0 * I2 + cx * sx + cy * sy
        Uk = expm(-1j * dt * H)
        U = Uk @ U
    return U

def nsr_and_energy(params, T):
    dt = T / K
    U = build_U(params, T)
    diff = Ud - U
    nsr = np.trace(diff.conj().T @ diff).real / Ud_norm2
    energy = np.sum(params**2) * dt
    return nsr, energy

def objective(params, T, budget, lam):
    nsr, energy = nsr_and_energy(params, T)
    return nsr + lam * (energy - budget) ** 2

def best_nsr_for_T(T, budget, lam=80.0, ntrials=6, seed0=0):
    best = None
    for trial in range(ntrials):
        rng = np.random.default_rng(seed0 + trial)
        x0 = rng.normal(scale=1.0, size=3*K)
        res = minimize(objective, x0, args=(T, budget, lam), method="L-BFGS-B",
                        options={"maxiter": 400})
        nsr, energy = nsr_and_energy(res.x, T)
        # keep only reasonably budget-respecting solutions, take min nsr
        if best is None or nsr < best[0]:
            best = (nsr, energy, res.x)
    return best

# ---------------------------------------------------------------
# 4. Sweep over T
# ---------------------------------------------------------------
budget = 3.0
Ts = np.concatenate([np.linspace(0.05, 0.15, 4, endpoint=False),
                      np.linspace(0.15, 2.2, 24),
                      np.linspace(2.4, 8.0, 10)])
nsr_vals = []
energy_vals = []
for T in Ts:
    nsr, energy, xbest = best_nsr_for_T(T, budget)
    nsr_vals.append(nsr)
    energy_vals.append(energy)
    print(f"T={T:6.3f}   NSR={nsr:8.5f}   energy={energy:6.3f}")

import os
save_dir = os.path.expanduser("~/bornscattering_results")
os.makedirs(save_dir, exist_ok=True)

np.save(os.path.join(save_dir, "Ts.npy"), Ts)
np.save(os.path.join(save_dir, "nsr_vals.npy"), np.array(nsr_vals))
np.save(os.path.join(save_dir, "energy_vals.npy"), np.array(energy_vals))
print(f"\nSaved arrays to {save_dir}")

Channel v1a (isotropic, a=1.0):   d=15.7496  g=5.7940
Channel v1b (isotropic, a=0.5):   d=1.9687  g=1.5332
Channel v2  (dipole,   a=1.0):    d=0.0000  g=5.793957705500554j

V1a =
 [[15.74960995+0.j  5.79395771+0.j]
 [ 5.79395771+0.j 15.74960995+0.j]]
V1b =
 [[1.96870124+0.j 1.53322607+0.j]
 [1.53322607+0.j 1.96870124+0.j]]
V2  =
 [[-0.+0.j          0.+5.79395771j]
 [-0.-5.79395771j -0.+0.j        ]]

Calibration matrix M (c0,cx) = M (eps_1a,eps_1b):
 [[15.74960995  1.96870124]
 [ 5.79395771  1.53322607]] 
det(M)= 12.741140820790314
T= 0.050   NSR= 1.89531   energy= 3.000
T= 0.075   NSR= 1.84402   energy= 3.000
T= 0.100   NSR= 1.79346   energy= 3.000
T= 0.125   NSR= 1.74362   energy= 3.001
T= 0.150   NSR= 1.69453   energy= 3.001
T= 0.239   NSR= 1.52564   energy= 3.001
T= 0.328   NSR= 1.36651   energy= 3.001
T= 0.417   NSR= 1.21723   energy= 3.001
T= 0.507   NSR= 1.07779   energy= 3.002
T= 0.596   NSR= 0.94810   energy= 3.002
T= 0.685   NSR= 0.82802   energy= 3.002
T= 0.774   NSR= 0.7173

In [4]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os

save_dir = os.path.expanduser("~/bornscattering_results")
Ts = np.load(os.path.join(save_dir, "Ts.npy"))
nsr = np.load(os.path.join(save_dir, "nsr_vals.npy"))

plt.rcParams.update({
    "font.size": 11,
    "axes.linewidth": 0.9,
})

fig, ax = plt.subplots(figsize=(4.6, 3.4))
ax.plot(Ts, nsr, color="#1f4e8c", linewidth=1.8, marker="o", markersize=3.2,
        markerfacecolor="white", markeredgewidth=0.9)
ax.set_xlabel(r"design time $T$")
ax.set_ylabel(r"NSR$(T)$")
ax.set_xlim(0, 8)
ax.set_ylim(-0.05, 2.0)
ax.grid(True, linewidth=0.4, alpha=0.6)
fig.tight_layout()
fig.savefig("figs/nsr_vs_T_born.pdf")
fig.savefig("figs/nsr_vs_T_born.png", dpi=200)
print("saved")

saved
